# Pseudo-Label: Sticky lock: pseudo-labels with p_max>=0.9 and others<0.3 (2 consecutive checks) become permanent
### Sticky Pseudo-Label Locking for Extreme Label Scarcity ($n=30$)

This notebook evaluates sticky pseudo-label locking against standard `pl`. When an unlabeled sample satisfies $p_{\max} \ge 0.9$ and others $< 0.3$ across 2 consecutive checks (`patience=2`), its pseudo-label is permanently locked and treated as fixed target for all subsequent epochs.
- **Idea**: Prevent label flipping and drift under high-uncertainty regimes by locking consistently confident predictions.
- **Modes Compared**: `pl` vs `lock`.

In [1]:
# Standalone Colab snippet:
# !git clone https://github.com/marcsanchezsala/semi-supervised-mnist.git
# %cd semi-supervised-mnist/05_sticky_lock
import os
import sys
import json
import numpy as np
import torch

if os.path.isdir("05_sticky_lock"):
    os.chdir("05_sticky_lock")
if "../common" not in sys.path:
    sys.path.append("../common")

from pl import load_mnist, make_splits, train, test_error, device

print(f"Using device: {device}")

Using device: cuda
Using device: cuda


In [2]:
X_train, y_train, X_test, y_test = load_mnist()
print(f"Loaded MNIST: X_train {X_train.shape}, X_test {X_test.shape}")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.19MB/s]

100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]

100%|██████████| 1.65M/1.65M [00:01<00:00, 1.29MB/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]



Loaded MNIST: X_train torch.Size([60000, 784]), X_test torch.Size([10000, 784])
Loaded MNIST: X_train torch.Size([60000, 784]), X_test torch.Size([10000, 784])


In [3]:
N_LABELED = 30
N_SEEDS = 3  # shortcut: 3 paired seeds (same split + init per mode), upgrade to 10 for tighter CI
MODES = ["pl", "lock"]
results = {m: [] for m in MODES}

for s in range(N_SEEDS):
    seed = 42 + s
    X_l, y_l, X_u, y_u, X_val, y_val = make_splits(X_train, y_train, n_labeled=N_LABELED, seed=seed)

    for mode in MODES:
        torch.manual_seed(seed)
        model, info = train(
            X_l, y_l, X_u, X_val, y_val, X_test, y_test,
            mode=mode, y_u=y_u, verbose=False
        )
        info["seed"] = seed
        info["mode"] = mode
        info["test_val_sel"] = test_error(model, X_test, y_test)
        results[mode].append(info)

        print(
            f"Seed {seed} | Mode {mode:7s} | "
            f"Val-sel Test Err: {info['test_val_sel']:5.2f}% | "
            f"Last Test Err: {info['test_last']:5.2f}% | "
            f"Cov: {info['coverage']:5.2f}% | "
            f"Prec: {info['precision']:5.2f}% | "
            f"Time: {info['seconds']:5.1f}s | "
            f"Peak: {info['peak_mb']:6.1f} MB",
            flush=True,
        )

        with open("results.json", "w") as f:
            json.dump(results, f, indent=2, default=float)

Seed 42 | Mode pl      | Val-sel Test Err: 27.56% | Last Test Err: 27.88% | Cov:   nan% | Prec:   nan% | Time: 118.9s | Peak: 1051.3 MB
Seed 42 | Mode lock    | Val-sel Test Err: 25.85% | Last Test Err: 25.85% | Cov: 100.00% | Prec: 72.54% | Time: 110.6s | Peak: 1071.7 MB
Seed 43 | Mode pl      | Val-sel Test Err: 39.09% | Last Test Err: 41.89% | Cov:   nan% | Prec:   nan% | Time: 105.4s | Peak: 1066.5 MB
Seed 43 | Mode lock    | Val-sel Test Err: 39.09% | Last Test Err: 38.53% | Cov: 100.00% | Prec: 59.36% | Time: 110.6s | Peak: 1071.0 MB
Seed 44 | Mode pl      | Val-sel Test Err: 37.11% | Last Test Err: 37.43% | Cov:   nan% | Prec:   nan% | Time: 105.2s | Peak: 1067.3 MB
Seed 44 | Mode lock    | Val-sel Test Err: 35.22% | Last Test Err: 35.48% | Cov: 100.00% | Prec: 64.74% | Time: 110.7s | Peak: 1071.8 MB


In [4]:
print("=" * 90)
print(f"{'Mode':<10} | {'Val-sel Test Err':>18} | {'Last Test Err':>16} | {'Coverage':>10} | {'Precision':>10} | {'Time (s)':>10} | {'Peak (MB)':>10}")
print("-" * 90)

for mode in MODES:
    runs = results[mode]
    val_sel_errs = [r["test_val_sel"] for r in runs]
    last_errs = [r["test_last"] for r in runs]
    covs = [r["coverage"] for r in runs]
    precs = [r["precision"] for r in runs]
    secs = [r["seconds"] for r in runs]
    peaks = [r["peak_mb"] for r in runs]

    val_str = f"{np.mean(val_sel_errs):5.2f} ± {np.std(val_sel_errs):4.2f}%"
    last_str = f"{np.mean(last_errs):5.2f} ± {np.std(last_errs):4.2f}%"
    cov_str = f"{np.nanmean(covs):5.2f}%" if not np.isnan(covs).all() else "N/A"
    prec_str = f"{np.nanmean(precs):5.2f}%" if not np.isnan(precs).all() else "N/A"
    sec_str = f"{np.mean(secs):6.1f}s"
    peak_str = f"{np.mean(peaks):6.1f} MB"

    print(f"{mode:<10} | {val_str:>18} | {last_str:>16} | {cov_str:>10} | {prec_str:>10} | {sec_str:>10} | {peak_str:>10}")

print("=" * 90)

if "pl" in MODES and len(MODES) > 1:
    other_modes = [m for m in MODES if m != "pl"]
    pl_runs = results["pl"]
    for other in other_modes:
        other_runs = results[other]
        diff_val = [other_runs[i]["test_val_sel"] - pl_runs[i]["test_val_sel"] for i in range(len(pl_runs))]
        diff_last = [other_runs[i]["test_last"] - pl_runs[i]["test_last"] for i in range(len(pl_runs))]

        mean_d_val, std_d_val = np.mean(diff_val), np.std(diff_val)
        mean_d_last, std_d_last = np.mean(diff_last), np.std(diff_last)

        print(f"\nPaired per-seed difference ({other} - pl):")
        print(f"  Val-selected test err diff: {mean_d_val:+5.2f} ± {std_d_val:4.2f}%")
        print(f"  Last-epoch test err diff:   {mean_d_last:+5.2f} ± {std_d_last:4.2f}%")

        if mean_d_val < -std_d_val:
            verdict = f"improves over PL by > 1 paired std (diff={mean_d_val:+.2f}%, std={std_d_val:.2f}%)"
        else:
            verdict = f"does NOT improve over PL by > 1 paired std (diff={mean_d_val:+.2f}%, std={std_d_val:.2f}%)"
        print(f"Verdict: {other} {verdict}")

Mode       |   Val-sel Test Err |    Last Test Err |   Coverage |  Precision |   Time (s) |  Peak (MB)
------------------------------------------------------------------------------------------
pl         |      34.59 ± 5.03% |    35.73 ± 5.84% |        N/A |        N/A |     109.9s |  1061.7 MB
lock       |      33.39 ± 5.56% |    33.29 ± 5.40% |    100.00% |     65.54% |     110.7s |  1071.5 MB

Paired per-seed difference (lock - pl):
  Val-selected test err diff: -1.20 ± 0.85%
  Last-epoch test err diff:   -2.45 ± 0.65%
Verdict: lock improves over PL by > 1 paired std (diff=-1.20%, std=0.85%)
Mode       |   Val-sel Test Err |    Last Test Err |   Coverage |  Precision |   Time (s) |  Peak (MB)
------------------------------------------------------------------------------------------
pl         |      34.59 ± 5.03% |    35.73 ± 5.84% |        N/A |        N/A |     109.9s |  1061.7 MB
lock       |      33.39 ± 5.56% |    33.29 ± 5.40% |    100.00% |     65.54% |     110.7s |  1071.5 